# GWM-RNN Training on Kaggle - PubMed Dataset

Train the lightweight GWM-RNN model for link prediction on PubMed medical citation network.

**Dataset: PubMed**
- ~20,000 nodes (medical research papers)
- ~7x larger than Cora dataset
- Medical domain with specialized terminology

**Model Advantages:**
- 🚀 **Fast**: 100x faster than LLM-based models
- 💾 **Lightweight**: ~10-20M parameters (vs 3-8B for LLMs)
- 💰 **Efficient**: Trains on consumer GPUs or even CPU
- 📊 **Competitive**: Achieves strong performance on graph tasks

**Training Time:** ~1-2 hours for all experiments on P100 GPU

---

## 1. Install Dependencies

In [ ]:
import os
import sys

# Check environment
IS_KAGGLE = os.path.exists('/kaggle')
print(f"Running on Kaggle: {IS_KAGGLE}")

if IS_KAGGLE:
    import torch
    print(f"PyTorch version: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

print("✓ Environment setup complete")

In [ ]:
# Install required packages
!pip install -q torch scikit-learn tqdm matplotlib seaborn

print("✓ All dependencies installed")

## 2. Configuration

Configure paths and training parameters for PubMed dataset. We'll train multiple configurations:
- **3 Pooling Methods**: last, mean, max
- **Multiple Hyperparameter Sets**: Different hidden dimensions and learning rates

In [ ]:
# ==============================================================================
# DATA PATHS CONFIGURATION
# ==============================================================================
if IS_KAGGLE:
    # Kaggle input paths for PubMed dataset
    DATA_DIR = '/kaggle/input/gwm-rnn-linkpred-pubmed'  # Your processed PubMed data
    OUTPUT_BASE_DIR = '/kaggle/working/experiments'
else:
    # Local paths
    DATA_DIR = 'data/pubmed/processed/gwm-rnn'
    OUTPUT_BASE_DIR = './trained/gwm-rnn/pubmed/experiments'

# ==============================================================================
# POOLING METHODS TO TEST
# ==============================================================================
POOLING_METHODS = ['last', 'mean', 'max']

# ==============================================================================
# HYPERPARAMETER CONFIGURATIONS
# ==============================================================================
# Define multiple configurations to test
HYPERPARAMETER_SETS = [
    # Standard configuration (from Cora analysis - best performing)
    {
        'name': 'standard',
        'hidden_dim': 256,
        'num_lstm_layers': 2,
        'dropout': 0.1,
        'learning_rate': 1e-3,
        'batch_size': 512,
        'description': 'Standard config (proven on Cora)'
    },
    # Larger model (may benefit from more data in PubMed)
    {
        'name': 'large',
        'hidden_dim': 512,
        'num_lstm_layers': 2,
        'dropout': 0.15,
        'learning_rate': 8e-4,
        'batch_size': 256,
        'description': 'Larger hidden dimension for bigger dataset'
    },
    # Deeper model
    {
        'name': 'deep',
        'hidden_dim': 256,
        'num_lstm_layers': 3,
        'dropout': 0.2,
        'learning_rate': 8e-4,
        'batch_size': 512,
        'description': 'Deeper network (3 layers)'
    },
    # Conservative (prevent overfitting on larger dataset)
    {
        'name': 'conservative',
        'hidden_dim': 128,
        'num_lstm_layers': 2,
        'dropout': 0.25,
        'learning_rate': 5e-4,
        'batch_size': 512,
        'description': 'Smaller, more regularized for PubMed'
    },
]

# ==============================================================================
# TRAINING PARAMETERS (FIXED ACROSS ALL EXPERIMENTS)
# ==============================================================================
NUM_EPOCHS = 100
WEIGHT_DECAY = 1e-4
MAX_GRAD_NORM = 1.0
EARLY_STOPPING_PATIENCE = 5
SEED = 42
NUM_WORKERS = 2

# ==============================================================================
# EXPERIMENT SELECTION
# ==============================================================================
# Choose which experiments to run
RUN_ALL_CONFIGS = False  # Set True to run all hyperparameter sets
SELECTED_CONFIG = 'standard'  # Which config to use if RUN_ALL_CONFIGS=False

print("="*80)
print(" "*15 + "GWM-RNN EXPERIMENT CONFIGURATION - PUBMED")
print("="*80)
print(f"\n📊 Dataset: PubMed (Medical Citation Network)")
print(f"   ~20,000 nodes (medical research papers)")
print(f"   ~7x larger than Cora dataset")
print(f"   Domain: Medical research citations")

print(f"\n📁 Data Directory: {DATA_DIR}")
print(f"📁 Output Base Directory: {OUTPUT_BASE_DIR}")

print(f"\n🔄 Pooling Methods to Test ({len(POOLING_METHODS)}):")
for pooling in POOLING_METHODS:
    print(f"   • {pooling}")

print(f"\n⚙️  Hyperparameter Sets Available ({len(HYPERPARAMETER_SETS)}):")
for i, config in enumerate(HYPERPARAMETER_SETS, 1):
    print(f"   {i}. {config['name']:15s} - {config['description']}")
    print(f"      Hidden: {config['hidden_dim']}, Layers: {config['num_lstm_layers']}, "
          f"LR: {config['learning_rate']}, Batch: {config['batch_size']}")

if RUN_ALL_CONFIGS:
    print(f"\n🚀 Mode: Running ALL configurations")
    print(f"   Total experiments: {len(POOLING_METHODS)} pooling × {len(HYPERPARAMETER_SETS)} configs = {len(POOLING_METHODS) * len(HYPERPARAMETER_SETS)} experiments")
    print(f"   Expected time: ~2-3 hours on P100 GPU")
else:
    print(f"\n🎯 Mode: Running SELECTED configuration only")
    print(f"   Config: {SELECTED_CONFIG}")
    print(f"   Total experiments: {len(POOLING_METHODS)} pooling × 1 config = {len(POOLING_METHODS)} experiments")
    print(f"   Expected time: ~30-45 minutes on P100 GPU")

print(f"\n⏱️  Fixed Parameters:")
print(f"   Epochs: {NUM_EPOCHS}")
print(f"   Weight decay: {WEIGHT_DECAY}")
print(f"   Gradient clipping: {MAX_GRAD_NORM}")
print(f"   Early stopping patience: {EARLY_STOPPING_PATIENCE}")
print(f"   Seed: {SEED}")

print(f"\n💡 Note: PubMed training is ~2-3x slower than Cora due to dataset size")
print("="*80)

## 3. Copy Training Files from GitHub

Clone repository and copy training scripts.

In [ ]:
required_files = ['model.py', 'dataset.py', 'inference.py', 'train.py', 'utils.py']

if IS_KAGGLE:
    print("="*70)
    print("Cloning GitHub repository...")
    print("="*70)
    
    # Clone your GitHub repo
    GITHUB_REPO = "https://github.com/HiIamPhuc/GWM.git"
    BRANCH = "main"
    
    !git clone {GITHUB_REPO} /kaggle/working/gwm
    %cd /kaggle/working/gwm
    !git checkout {BRANCH}
    !git pull
    %cd ../
    
    # Copy files from repo to working directory
    repo_path = "/kaggle/working/gwm/gwm-rnn/link-prediction"
    
    print(f"\nCopying files from {repo_path}...")
    for file in required_files:
        !cp {repo_path}/{file} /kaggle/working/
        print(f"✓ Copied {file}")
else:
    print("Running locally - files should be in current directory")

# Verify files exist
import os
missing_files = [f for f in required_files if not os.path.exists(f)]

if missing_files:
    print(f"\n❌ Missing files: {missing_files}")
    raise FileNotFoundError(f"Required files not found: {missing_files}")
else:
    print(f"\n✓ All required files ready: {required_files}")

## 4. Run Training Experiments

Train models with all pooling methods and selected hyperparameter configurations on PubMed.

In [ ]:
import time
from datetime import datetime
import json
from pathlib import Path
import pandas as pd
import numpy as np

# Determine which configs to run
if RUN_ALL_CONFIGS:
    configs_to_run = HYPERPARAMETER_SETS
else:
    configs_to_run = [c for c in HYPERPARAMETER_SETS if c['name'] == SELECTED_CONFIG]

# Track all experiment results
all_results = []
experiment_start_time = time.time()

print("="*80)
print(" "*20 + "STARTING PUBMED EXPERIMENTS")
print("="*80)
print(f"\nTotal experiments to run: {len(POOLING_METHODS)} × {len(configs_to_run)} = {len(POOLING_METHODS) * len(configs_to_run)}")
print(f"Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"⏱️  Expected time: ~{len(POOLING_METHODS) * len(configs_to_run) * 15:.0f}-{len(POOLING_METHODS) * len(configs_to_run) * 20:.0f} minutes")
print("="*80)

experiment_num = 0
total_experiments = len(POOLING_METHODS) * len(configs_to_run)

for config in configs_to_run:
    for pooling in POOLING_METHODS:
        experiment_num += 1
        
        print(f"\n{'='*80}")
        print(f" EXPERIMENT {experiment_num}/{total_experiments}: {config['name'].upper()} + {pooling.upper()}-POOLING (PubMed)")
        print(f"{'='*80}")
        
        # Create output directory for this experiment
        output_dir = f"{OUTPUT_BASE_DIR}/{config['name']}/{pooling}-pooling"
        
        # Build training command
        cmd = f"""python train.py \\
            --data_dir {DATA_DIR} \\
            --output_dir {output_dir} \\
            --hidden_dim {config['hidden_dim']} \\
            --num_lstm_layers {config['num_lstm_layers']} \\
            --dropout {config['dropout']} \\
            --pooling {pooling} \\
            --num_epochs {NUM_EPOCHS} \\
            --batch_size {config['batch_size']} \\
            --learning_rate {config['learning_rate']} \\
            --weight_decay {WEIGHT_DECAY} \\
            --max_grad_norm {MAX_GRAD_NORM} \\
            --early_stopping_patience {EARLY_STOPPING_PATIENCE} \\
            --seed {SEED} \\
            --num_workers {NUM_WORKERS}"""
        
        print(f"\n📋 Configuration:")
        print(f"   Dataset: PubMed (~20,000 nodes)")
        print(f"   Config: {config['name']} - {config['description']}")
        print(f"   Pooling: {pooling}")
        print(f"   Hidden dim: {config['hidden_dim']}")
        print(f"   LSTM layers: {config['num_lstm_layers']}")
        print(f"   Dropout: {config['dropout']}")
        print(f"   Learning rate: {config['learning_rate']}")
        print(f"   Batch size: {config['batch_size']}")
        print(f"   Output: {output_dir}")
        
        print(f"\n🚀 Starting training...")
        print("-"*80)
        
        # Execute training
        !{cmd}
        
        # Load and store results
        try:
            result_path = Path(output_dir) / "test_results.json"
            history_path = Path(output_dir) / "training_history.json"
            
            if result_path.exists() and history_path.exists():
                with open(result_path) as f:
                    test_results = json.load(f)
                with open(history_path) as f:
                    history = json.load(f)
                
                # Calculate training time
                train_time = sum(h['epoch_time'] for h in history)
                
                # Store result
                all_results.append({
                    'config_name': config['name'],
                    'pooling': pooling,
                    'hidden_dim': config['hidden_dim'],
                    'num_layers': config['num_lstm_layers'],
                    'dropout': config['dropout'],
                    'learning_rate': config['learning_rate'],
                    'batch_size': config['batch_size'],
                    'test_accuracy': test_results['accuracy'],
                    'test_f1': test_results['f1'],
                    'test_auc': test_results['auc'],
                    'test_precision': test_results['precision'],
                    'test_recall': test_results['recall'],
                    'training_time': train_time,
                    'total_epochs': len(history),
                    'output_dir': output_dir
                })
                
                print(f"\n✅ Experiment {experiment_num} completed successfully!")
                print(f"   Test Accuracy: {test_results['accuracy']:.4f} ({test_results['accuracy']*100:.2f}%)")
                print(f"   Test F1: {test_results['f1']:.4f}")
                print(f"   Training time: {train_time:.1f}s ({train_time/60:.1f} min)")
            else:
                print(f"\n⚠️  Results not found for experiment {experiment_num}")
        except Exception as e:
            print(f"\n❌ Error loading results for experiment {experiment_num}: {e}")
        
        print(f"\n{'='*80}\n")

total_time = time.time() - experiment_start_time

print(f"\n{'='*80}")
print(" "*20 + "ALL PUBMED EXPERIMENTS COMPLETED")
print(f"{'='*80}")
print(f"Total experiments: {len(all_results)}/{total_experiments}")
print(f"Total time: {total_time/60:.1f} minutes ({total_time/3600:.2f} hours)")
print(f"Average time per experiment: {total_time/len(all_results)/60:.1f} minutes")
print(f"Completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"{'='*80}\n")

## 5. Comprehensive Results Analysis

Analyze and compare all PubMed experiments across pooling methods and hyperparameter sets.

In [ ]:
if len(all_results) > 0:
    # Create DataFrame
    df_results = pd.DataFrame(all_results)
    
    print("="*80)
    print(" "*15 + "PUBMED EXPERIMENT RESULTS SUMMARY")
    print("="*80)
    
    # Overall best
    best_idx = df_results['test_accuracy'].idxmax()
    best_result = df_results.loc[best_idx]
    
    print(f"\n🏆 BEST OVERALL PERFORMANCE:")
    print(f"   Config: {best_result['config_name']} + {best_result['pooling']}-pooling")
    print(f"   Test Accuracy: {best_result['test_accuracy']:.4f} ({best_result['test_accuracy']*100:.2f}%)")
    print(f"   Test F1: {best_result['test_f1']:.4f}")
    print(f"   Test AUC: {best_result['test_auc']:.4f}")
    print(f"   Training time: {best_result['training_time']:.1f}s ({best_result['training_time']/60:.1f} min)")
    
    # Pooling method comparison
    print(f"\n📊 PERFORMANCE BY POOLING METHOD:")
    print("-"*80)
    pooling_summary = df_results.groupby('pooling').agg({
        'test_accuracy': ['mean', 'std', 'max'],
        'test_f1': ['mean', 'max'],
        'test_auc': ['mean', 'max'],
        'training_time': 'mean'
    }).round(4)
    
    for pooling in POOLING_METHODS:
        if pooling in pooling_summary.index:
            row = pooling_summary.loc[pooling]
            print(f"\n{pooling.upper()}-Pooling:")
            print(f"   Avg Accuracy: {row[('test_accuracy', 'mean')]:.4f} ± {row[('test_accuracy', 'std')]:.4f}")
            print(f"   Max Accuracy: {row[('test_accuracy', 'max')]:.4f}")
            print(f"   Avg F1: {row[('test_f1', 'mean')]:.4f}")
            print(f"   Avg AUC: {row[('test_auc', 'mean')]:.4f}")
            print(f"   Avg Training Time: {row[('training_time', 'mean')]/60:.1f} min")
    
    # Configuration comparison
    if len(configs_to_run) > 1:
        print(f"\n⚙️  PERFORMANCE BY CONFIGURATION:")
        print("-"*80)
        config_summary = df_results.groupby('config_name').agg({
            'test_accuracy': ['mean', 'std', 'max'],
            'test_f1': ['mean', 'max'],
            'training_time': 'mean'
        }).round(4)
        
        for config_name in [c['name'] for c in configs_to_run]:
            if config_name in config_summary.index:
                row = config_summary.loc[config_name]
                config_desc = [c['description'] for c in configs_to_run if c['name'] == config_name][0]
                print(f"\n{config_name.upper()} ({config_desc}):")
                print(f"   Avg Accuracy: {row[('test_accuracy', 'mean')]:.4f} ± {row[('test_accuracy', 'std')]:.4f}")
                print(f"   Max Accuracy: {row[('test_accuracy', 'max')]:.4f}")
                print(f"   Avg F1: {row[('test_f1', 'mean')]:.4f}")
                print(f"   Avg Training Time: {row[('training_time', 'mean')]/60:.1f} min")
    
    # Detailed results table
    print(f"\n📋 DETAILED RESULTS TABLE:")
    print("-"*80)
    display_cols = ['config_name', 'pooling', 'test_accuracy', 'test_f1', 'test_auc', 
                    'hidden_dim', 'num_layers', 'learning_rate', 'training_time']
    df_display = df_results[display_cols].copy()
    df_display['test_accuracy'] = df_display['test_accuracy'].apply(lambda x: f"{x:.4f}")
    df_display['test_f1'] = df_display['test_f1'].apply(lambda x: f"{x:.4f}")
    df_display['test_auc'] = df_display['test_auc'].apply(lambda x: f"{x:.4f}")
    df_display['learning_rate'] = df_display['learning_rate'].apply(lambda x: f"{x:.1e}")
    df_display['training_time'] = df_display['training_time'].apply(lambda x: f"{x/60:.1f}min")
    
    print(df_display.to_string(index=False))
    
    # Save results
    results_csv_path = f"{OUTPUT_BASE_DIR}/pubmed_all_results_summary.csv"
    df_results.to_csv(results_csv_path, index=False)
    print(f"\n✓ Saved detailed results to: {results_csv_path}")
    
    print("\n" + "="*80)
else:
    print("❌ No results available. Please run training experiments first.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(all_results) > 0:
    df_results = pd.DataFrame(all_results)
    
    # Set style
    sns.set_style("whitegrid")
    
    # Create comprehensive visualization
    fig = plt.figure(figsize=(18, 12))
    fig.suptitle('GWM-RNN Performance on PubMed Dataset', fontsize=16, fontweight='bold')
    
    # 1. Accuracy comparison by pooling method
    ax1 = plt.subplot(2, 3, 1)
    pooling_data = df_results.groupby('pooling')['test_accuracy'].apply(list)
    bp1 = ax1.boxplot([pooling_data[p] for p in POOLING_METHODS if p in pooling_data.index],
                       labels=[p.upper() for p in POOLING_METHODS if p in pooling_data.index],
                       patch_artist=True)
    for patch in bp1['boxes']:
        patch.set_facecolor('lightblue')
    ax1.set_ylabel('Test Accuracy', fontsize=11, fontweight='bold')
    ax1.set_title('Accuracy by Pooling Method', fontsize=12, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    
    # 2. F1 score comparison
    ax2 = plt.subplot(2, 3, 2)
    pooling_f1 = df_results.groupby('pooling')['test_f1'].apply(list)
    bp2 = ax2.boxplot([pooling_f1[p] for p in POOLING_METHODS if p in pooling_f1.index],
                       labels=[p.upper() for p in POOLING_METHODS if p in pooling_f1.index],
                       patch_artist=True)
    for patch in bp2['boxes']:
        patch.set_facecolor('lightgreen')
    ax2.set_ylabel('Test F1 Score', fontsize=11, fontweight='bold')
    ax2.set_title('F1 Score by Pooling Method', fontsize=12, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    
    # 3. AUC comparison
    ax3 = plt.subplot(2, 3, 3)
    pooling_auc = df_results.groupby('pooling')['test_auc'].apply(list)
    bp3 = ax3.boxplot([pooling_auc[p] for p in POOLING_METHODS if p in pooling_auc.index],
                       labels=[p.upper() for p in POOLING_METHODS if p in pooling_auc.index],
                       patch_artist=True)
    for patch in bp3['boxes']:
        patch.set_facecolor('lightcoral')
    ax3.set_ylabel('Test AUC', fontsize=11, fontweight='bold')
    ax3.set_title('AUC by Pooling Method', fontsize=12, fontweight='bold')
    ax3.grid(True, alpha=0.3)
    
    # 4. Scatter: Accuracy vs Training Time
    ax4 = plt.subplot(2, 3, 4)
    colors = {'last': 'blue', 'mean': 'green', 'max': 'red'}
    for pooling in POOLING_METHODS:
        mask = df_results['pooling'] == pooling
        ax4.scatter(df_results[mask]['training_time']/60, 
                   df_results[mask]['test_accuracy'],
                   c=colors.get(pooling, 'gray'),
                   label=pooling.upper(),
                   s=100, alpha=0.6, edgecolors='black')
    ax4.set_xlabel('Training Time (minutes)', fontsize=11, fontweight='bold')
    ax4.set_ylabel('Test Accuracy', fontsize=11, fontweight='bold')
    ax4.set_title('Accuracy vs Training Time', fontsize=12, fontweight='bold')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    
    # 5. Bar chart: Best result per pooling method
    ax5 = plt.subplot(2, 3, 5)
    best_per_pooling = df_results.groupby('pooling')['test_accuracy'].max()
    bars = ax5.bar(range(len(best_per_pooling)), best_per_pooling.values, 
                   color=['blue', 'red', 'green'][:len(best_per_pooling)])
    ax5.set_xticks(range(len(best_per_pooling)))
    ax5.set_xticklabels([p.upper() for p in best_per_pooling.index])
    ax5.set_ylabel('Best Test Accuracy', fontsize=11, fontweight='bold')
    ax5.set_title('Best Accuracy per Pooling (PubMed)', fontsize=12, fontweight='bold')
    ax5.set_ylim([best_per_pooling.min() - 0.02, best_per_pooling.max() + 0.01])
    
    # Add value labels on bars
    for i, (bar, val) in enumerate(zip(bars, best_per_pooling.values)):
        ax5.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                f'{val:.4f}\n({val*100:.2f}%)',
                ha='center', va='bottom', fontweight='bold', fontsize=9)
    ax5.grid(True, alpha=0.3, axis='y')
    
    # 6. Heatmap or Precision-Recall
    if len(configs_to_run) > 1:
        ax6 = plt.subplot(2, 3, 6)
        pivot_table = df_results.pivot_table(
            values='test_accuracy',
            index='config_name',
            columns='pooling',
            aggfunc='mean'
        )
        sns.heatmap(pivot_table, annot=True, fmt='.4f', cmap='YlOrRd', 
                   ax=ax6, cbar_kws={'label': 'Test Accuracy'})
        ax6.set_title('Accuracy Heatmap: Config × Pooling', fontsize=12, fontweight='bold')
        ax6.set_xlabel('Pooling Method', fontsize=11, fontweight='bold')
        ax6.set_ylabel('Configuration', fontsize=11, fontweight='bold')
    else:
        ax6 = plt.subplot(2, 3, 6)
        for pooling in POOLING_METHODS:
            mask = df_results['pooling'] == pooling
            if mask.any():
                ax6.scatter(df_results[mask]['test_recall'], 
                          df_results[mask]['test_precision'],
                          c=colors.get(pooling, 'gray'),
                          label=pooling.upper(),
                          s=150, alpha=0.6, edgecolors='black')
        ax6.set_xlabel('Test Recall', fontsize=11, fontweight='bold')
        ax6.set_ylabel('Test Precision', fontsize=11, fontweight='bold')
        ax6.set_title('Precision-Recall Trade-off (PubMed)', fontsize=12, fontweight='bold')
        ax6.legend()
        ax6.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    # Save figure
    viz_path = f"{OUTPUT_BASE_DIR}/pubmed_comprehensive_comparison.png"
    plt.savefig(viz_path, dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"✓ Saved comprehensive visualization to: {viz_path}")
else:
    print("❌ No results to visualize.")

## 6. Best Model Analysis & Recommendations

Detailed analysis of the best performing model

In [ ]:
if len(all_results) > 0:
    df_results = pd.DataFrame(all_results)
    
    print("=" * 80)
    print("BEST MODEL ANALYSIS ON PUBMED DATASET")
    print("=" * 80)
    print()
    
    # Find best model
    best_idx = df_results['test_accuracy'].idxmax()
    best_result = df_results.iloc[best_idx]
    
    print("🏆 BEST MODEL CONFIGURATION:")
    print(f"  • Configuration: {best_result['config_name'].upper()}")
    print(f"  • Pooling Method: {best_result['pooling'].upper()}")
    print(f"  • Hidden Dimension: {best_result['hidden_dim']}")
    print(f"  • LSTM Layers: {best_result['num_layers']}")
    print(f"  • Dropout: {best_result['dropout']:.2f}")
    print(f"  • Learning Rate: {best_result['learning_rate']}")
    print(f"  • Batch Size: {best_result['batch_size']}")
    print()
    
    print("📊 PERFORMANCE METRICS:")
    print(f"  • Test Accuracy:  {best_result['test_accuracy']:.4f} ({best_result['test_accuracy']*100:.2f}%)")
    print(f"  • Test F1 Score:  {best_result['test_f1']:.4f} ({best_result['test_f1']*100:.2f}%)")
    print(f"  • Test Precision: {best_result['test_precision']:.4f} ({best_result['test_precision']*100:.2f}%)")
    print(f"  • Test Recall:    {best_result['test_recall']:.4f} ({best_result['test_recall']*100:.2f}%)")
    print(f"  • Test AUC-ROC:   {best_result['test_auc']:.4f} ({best_result['test_auc']*100:.2f}%)")
    print()
    
    print("⏱️  TRAINING EFFICIENCY:")
    print(f"  • Training Time:  {best_result['training_time']:.1f}s ({best_result['training_time']/60:.1f} min)")
    print(f"  • Best Epoch:     {best_result['best_epoch']}")
    print(f"  • Total Epochs:   {best_result['total_epochs']}")
    print()
    
    # Model size estimation (simplified)
    hidden_dim = best_result['hidden_dim']
    num_layers = best_result['num_layers']
    input_dim = 1433  # PubMed feature dimension (same as Cora)
    
    # Approximate parameter count
    # LSTM: 4 * (input_dim * hidden_dim + hidden_dim^2 + hidden_dim) for first layer
    # Then: 4 * (hidden_dim * hidden_dim + hidden_dim^2 + hidden_dim) for each subsequent layer
    # Classifier: hidden_dim * 2
    lstm_params_first = 4 * (input_dim * hidden_dim + hidden_dim * hidden_dim + hidden_dim)
    lstm_params_rest = (num_layers - 1) * 4 * (hidden_dim * hidden_dim + hidden_dim * hidden_dim + hidden_dim)
    classifier_params = hidden_dim * 2 + 2  # +2 for bias
    total_params = lstm_params_first + lstm_params_rest + classifier_params
    
    print("🔢 MODEL SIZE:")
    print(f"  • Total Parameters:     ~{total_params:,}")
    print(f"  • Model Size:           ~{total_params * 4 / (1024**2):.2f} MB (float32)")
    print()
    print("  📝 COMPARISON WITH LLM BASELINES:")
    print(f"  • GWM-RNN (this model): {total_params:,} parameters")
    print(f"  • Typical LLM (e.g., 3B): 3,000,000,000 parameters")
    print(f"  • Parameter Ratio:      1 : {3_000_000_000 / total_params:.0f}")
    print(f"  • This model is {3_000_000_000 / total_params:.0f}× more parameter-efficient!")
    print()
    
    # Ranking analysis
    print("📈 POOLING METHOD RANKING (by average accuracy on PubMed):")
    pooling_ranking = df_results.groupby('pooling').agg({
        'test_accuracy': 'mean',
        'test_f1': 'mean',
        'test_auc': 'mean',
        'training_time': 'mean'
    }).sort_values('test_accuracy', ascending=False)
    
    for rank, (pooling, row) in enumerate(pooling_ranking.iterrows(), 1):
        print(f"  {rank}. {pooling.upper()}-pooling:")
        print(f"     • Avg Accuracy: {row['test_accuracy']:.4f} ({row['test_accuracy']*100:.2f}%)")
        print(f"     • Avg F1 Score: {row['test_f1']:.4f}")
        print(f"     • Avg AUC:      {row['test_auc']:.4f}")
        print(f"     • Avg Time:     {row['training_time']/60:.1f} min")
    print()
    
    # Configuration ranking (if multiple configs)
    if len(configs_to_run) > 1:
        print("📊 CONFIGURATION RANKING (by average accuracy on PubMed):")
        config_ranking = df_results.groupby('config_name').agg({
            'test_accuracy': 'mean',
            'test_f1': 'mean',
            'test_auc': 'mean',
            'training_time': 'mean'
        }).sort_values('test_accuracy', ascending=False)
        
        for rank, (config, row) in enumerate(config_ranking.iterrows(), 1):
            print(f"  {rank}. {config.upper()} configuration:")
            print(f"     • Avg Accuracy: {row['test_accuracy']:.4f} ({row['test_accuracy']*100:.2f}%)")
            print(f"     • Avg F1 Score: {row['test_f1']:.4f}")
            print(f"     • Avg Time:     {row['training_time']/60:.1f} min")
        print()
    
    # Recommendations
    print("💡 RECOMMENDATIONS:")
    print("  For PRODUCTION deployment on PubMed-scale medical datasets:")
    best_pooling = pooling_ranking.index[0]
    print(f"  • Use {best_pooling.upper()}-pooling for best accuracy")
    print(f"  • Training time: ~{pooling_ranking.loc[best_pooling, 'training_time']/60:.1f} min (very efficient!)")
    print(f"  • Expected accuracy: ~{pooling_ranking.loc[best_pooling, 'test_accuracy']*100:.2f}%")
    
    if len(configs_to_run) > 1:
        best_config = config_ranking.index[0]
        print(f"  • Best configuration: {best_config.upper()}")
    
    print()
    print("  For RESEARCH experiments on medical citation networks:")
    print("  • All pooling methods show strong performance (>85% typical)")
    print("  • Training is efficient even on larger datasets (~15-20 min)")
    print("  • Model remains lightweight and deployable")
    print()
    print("=" * 80)
else:
    print("❌ No results available for best model analysis.")

## 7. Training Curves for Best Model

Detailed training history and convergence analysis

In [ ]:
if len(all_results) > 0:
    df_results = pd.DataFrame(all_results)
    
    # Find best model
    best_idx = df_results['test_accuracy'].idxmax()
    best_result = df_results.iloc[best_idx]
    
    # Load training history
    history_path = f"{OUTPUT_BASE_DIR}/{best_result['config_name']}/{best_result['pooling']}-pooling/training_history.json"
    
    if os.path.exists(history_path):
        with open(history_path, 'r') as f:
            history = json.load(f)
        
        epochs = list(range(1, len(history['train_loss']) + 1))
        
        # Create training curves
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        fig.suptitle(f'Training Curves - Best Model ({best_result["pooling"].upper()}-pooling, {best_result["config_name"].upper()} config) on PubMed', 
                     fontsize=14, fontweight='bold')
        
        # 1. Loss curves
        ax1 = axes[0, 0]
        ax1.plot(epochs, history['train_loss'], label='Train Loss', linewidth=2, color='blue')
        ax1.plot(epochs, history['val_loss'], label='Val Loss', linewidth=2, color='orange')
        ax1.axvline(x=best_result['best_epoch'], color='red', linestyle='--', 
                   label=f'Best Epoch ({best_result["best_epoch"]})', alpha=0.7)
        ax1.set_xlabel('Epoch', fontweight='bold')
        ax1.set_ylabel('Loss', fontweight='bold')
        ax1.set_title('Training and Validation Loss', fontweight='bold')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        # 2. Accuracy curves
        ax2 = axes[0, 1]
        ax2.plot(epochs, history['train_accuracy'], label='Train Accuracy', linewidth=2, color='blue')
        ax2.plot(epochs, history['val_accuracy'], label='Val Accuracy', linewidth=2, color='orange')
        ax2.axvline(x=best_result['best_epoch'], color='red', linestyle='--', 
                   label=f'Best Epoch ({best_result["best_epoch"]})', alpha=0.7)
        ax2.set_xlabel('Epoch', fontweight='bold')
        ax2.set_ylabel('Accuracy', fontweight='bold')
        ax2.set_title('Training and Validation Accuracy', fontweight='bold')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        # 3. F1 Score and AUC curves
        ax3 = axes[1, 0]
        ax3.plot(epochs, history['val_f1'], label='Val F1 Score', linewidth=2, color='green')
        ax3.plot(epochs, history['val_auc'], label='Val AUC-ROC', linewidth=2, color='purple')
        ax3.axvline(x=best_result['best_epoch'], color='red', linestyle='--', 
                   label=f'Best Epoch ({best_result["best_epoch"]})', alpha=0.7)
        ax3.set_xlabel('Epoch', fontweight='bold')
        ax3.set_ylabel('Score', fontweight='bold')
        ax3.set_title('Validation F1 Score and AUC-ROC', fontweight='bold')
        ax3.legend()
        ax3.grid(True, alpha=0.3)
        
        # 4. Generalization gap (train - val accuracy)
        ax4 = axes[1, 1]
        gen_gap = [train - val for train, val in zip(history['train_accuracy'], history['val_accuracy'])]
        ax4.plot(epochs, gen_gap, linewidth=2, color='red', label='Generalization Gap')
        ax4.axhline(y=0, color='black', linestyle='-', alpha=0.3)
        ax4.axvline(x=best_result['best_epoch'], color='red', linestyle='--', 
                   label=f'Best Epoch ({best_result["best_epoch"]})', alpha=0.7)
        ax4.set_xlabel('Epoch', fontweight='bold')
        ax4.set_ylabel('Train Acc - Val Acc', fontweight='bold')
        ax4.set_title('Generalization Gap (Overfitting Analysis)', fontweight='bold')
        ax4.legend()
        ax4.grid(True, alpha=0.3)
        
        plt.tight_layout()
        
        # Save figure
        curves_path = f"{OUTPUT_BASE_DIR}/pubmed_best_model_training_curves.png"
        plt.savefig(curves_path, dpi=150, bbox_inches='tight')
        plt.show()
        
        print(f"✓ Saved training curves to: {curves_path}")
        print()
        print("📊 TRAINING STATISTICS:")
        print(f"  • Best Validation Accuracy: {max(history['val_accuracy']):.4f} at epoch {best_result['best_epoch']}")
        print(f"  • Final Train Accuracy:     {history['train_accuracy'][-1]:.4f}")
        print(f"  • Final Val Accuracy:       {history['val_accuracy'][-1]:.4f}")
        print(f"  • Final Generalization Gap: {gen_gap[-1]:.4f}")
        print(f"  • Best Val F1:              {max(history['val_f1']):.4f}")
        print(f"  • Best Val AUC:             {max(history['val_auc']):.4f}")
        print()
        
        # Convergence analysis
        if best_result['best_epoch'] < best_result['total_epochs'] * 0.5:
            print("  ✓ Model converged early (< 50% of total epochs) - efficient training!")
        elif best_result['best_epoch'] < best_result['total_epochs'] * 0.8:
            print("  ✓ Model converged normally (50-80% of total epochs)")
        else:
            print("  ⚠️  Model converged late (> 80% of total epochs) - consider more epochs")
        
        if abs(gen_gap[-1]) < 0.05:
            print("  ✓ Low generalization gap (< 5%) - no significant overfitting")
        elif abs(gen_gap[-1]) < 0.10:
            print("  ⚠️  Moderate generalization gap (5-10%) - slight overfitting")
        else:
            print("  ⚠️  High generalization gap (> 10%) - overfitting detected")
        
    else:
        print(f"❌ Training history not found at: {history_path}")
        print("   This may be expected if using the default train.py script.")
else:
    print("❌ No results available for training curves analysis.")

## 8. Download Results

List of all output files and download instructions

In [ ]:
print("=" * 80)
print("EXPERIMENT OUTPUT SUMMARY - PUBMED DATASET")
print("=" * 80)
print()
print(f"📁 Output directory: {OUTPUT_BASE_DIR}")
print()

# List experiment directories
print("📂 EXPERIMENT DIRECTORIES:")
for config_name in os.listdir(OUTPUT_BASE_DIR):
    config_path = os.path.join(OUTPUT_BASE_DIR, config_name)
    if os.path.isdir(config_path) and not config_name.startswith('.'):
        print(f"\n  • {config_name}/")
        for pooling_dir in os.listdir(config_path):
            pooling_path = os.path.join(config_path, pooling_dir)
            if os.path.isdir(pooling_path):
                print(f"    └── {pooling_dir}/")
                # List key files
                files = os.listdir(pooling_path)
                for file in sorted(files):
                    if file.endswith(('.pt', '.json', '.csv', '.png')):
                        file_path = os.path.join(pooling_path, file)
                        size_mb = os.path.getsize(file_path) / (1024 * 1024)
                        print(f"        ├── {file} ({size_mb:.2f} MB)")

print()
print("📊 SUMMARY FILES:")
summary_files = [
    'pubmed_all_results_summary.csv',
    'pubmed_comprehensive_comparison.png',
    'pubmed_best_model_training_curves.png'
]

for file in summary_files:
    file_path = os.path.join(OUTPUT_BASE_DIR, file)
    if os.path.exists(file_path):
        size_mb = os.path.getsize(file_path) / (1024 * 1024)
        print(f"  ✓ {file} ({size_mb:.2f} MB)")
    else:
        print(f"  ✗ {file} (not found)")

print()

# Best model info
if len(all_results) > 0:
    df_results = pd.DataFrame(all_results)
    best_idx = df_results['test_accuracy'].idxmax()
    best_result = df_results.iloc[best_idx]
    
    print("🏆 BEST MODEL FILES:")
    best_model_dir = f"{OUTPUT_BASE_DIR}/{best_result['config_name']}/{best_result['pooling']}-pooling"
    print(f"  Location: {best_model_dir}/")
    print(f"  • Model checkpoint: best_model.pt")
    print(f"  • Test predictions: test_predictions.csv")
    print(f"  • Test summary: test_summary.csv")
    print(f"  • Training history: training_history.json")
    print(f"  • Metrics: test_metrics.json")
    print()

print("💾 DOWNLOAD INSTRUCTIONS (Kaggle):")
print("  1. Click the 'Save Version' button (top right)")
print("  2. Select 'Save & Run All (Commit)'")
print("  3. After completion, go to the 'Output' tab")
print("  4. Download individual files or the entire output folder")
print()
print("  Alternative (via code):")
print(f"  !zip -r pubmed_experiments.zip {OUTPUT_BASE_DIR}")
print()

# Total size estimate
total_size = 0
for root, dirs, files in os.walk(OUTPUT_BASE_DIR):
    for file in files:
        file_path = os.path.join(root, file)
        total_size += os.path.getsize(file_path)

print(f"📦 Total output size: {total_size / (1024**2):.2f} MB")
print()
print("=" * 80)

# Experiment summary
if len(all_results) > 0:
    print()
    print("EXPERIMENT SUMMARY:")
    print(f"  • Total experiments run: {len(all_results)}")
    print(f"  • Pooling methods tested: {df_results['pooling'].nunique()}")
    print(f"  • Configurations tested: {df_results['config_name'].nunique()}")
    print(f"  • Best accuracy: {df_results['test_accuracy'].max():.4f} ({df_results['test_accuracy'].max()*100:.2f}%)")
    print(f"  • Best F1 score: {df_results['test_f1'].max():.4f}")
    print(f"  • Best AUC: {df_results['test_auc'].max():.4f}")
    print(f"  • Total training time: {df_results['training_time'].sum()/60:.1f} min")
    print("=" * 80)